Do decoding on single electrodes; pool results

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "5"  # Limit OpenMP
os.environ["MKL_NUM_THREADS"] = "5"  # Limit MKL (Intel Math Kernel Library)
os.environ["OPENBLAS_NUM_THREADS"] = "5"  # Limit OpenBLAS
os.environ["NUMEXPR_MAX_THREADS"] = "5"  # Limit NumExpr if installed

In [4]:
import itertools
from pathlib import Path
import re
from typing import Literal

from matplotlib.colors import CenteredNorm
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import pickle
from scipy.stats import ttest_ind
import seaborn as sns
import torch
# from tqdm.auto import tqdm
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.pipeline import make_pipeline

from src.data import get_electrode_df
from src.stimuli import POD_dict

In [188]:
outdir = "tmp_single_electrode"

In [6]:
all_epoch_paths = list(Path("outputs/epochs_preprocessed").glob("*.fif"))

In [ ]:
epochs = {}
for path in tqdm(all_epoch_paths):
    subject_name = re.findall("(EC[\d]+)_epo", str(path))[0]
    epochs[subject_name] = mne.read_epochs(str(path)).pick("ecog").resample(100)

In [ ]:
electrode_df = pd.concat([get_electrode_df(subject_name) for subject_name in epochs.keys()],
                         keys=epochs.keys(), names=["subject"])
electrode_df["roi"] = electrode_df.roi.astype(str)
electrode_df = electrode_df.droplevel("electrode_name")

# Drop electrodes metadata which don't have corresponding data
for subject, epochs_i in epochs.items():
    electrode_df.loc[subject, "keep"] = np.arange(len(electrode_df.loc[subject])) < len(epochs_i.info["ch_names"])
electrode_df = electrode_df[electrode_df["keep"]].drop(columns="keep")

electrode_df

## Find speech-responsive electrodes

In [9]:
# power threshold relative to pre-speech baseline which defines a "speech responsive" electrode
# if we see absolute value change >= this threshold, call the electrode speech-responsive
speech_responsive_threshold = 0.3

In [ ]:
# demo this
dd = next(iter(epochs.values())).copy().apply_baseline((None, 0)).average().crop(tmin=0, tmax=0.9).get_data()
keep = np.abs(dd).max(axis=1) > speech_responsive_threshold

f, ax = plt.subplots(figsize=(8, 4))
for line, k in zip(dd, keep):
    plt.plot(line, color="r" if k else "k", alpha=0.1)

In [ ]:
for subject, epochs_i in epochs.items():
    epochs_i = epochs_i.copy().apply_baseline((None, 0)).average().crop(tmin=0, tmax=0.9).get_data()
    assert epochs_i.ndim == 2
    speech_responsive_i = np.abs(epochs_i).max(axis=1) > speech_responsive_threshold

    if len(speech_responsive_i) > len(electrode_df.loc[subject]):
        speech_responsive_i = speech_responsive_i[:len(electrode_df.loc[subject])]
    electrode_df.loc[subject, "speech_responsive"] = speech_responsive_i

In [12]:
electrode_df = electrode_df.astype({"speech_responsive": bool})

In [197]:
def run_decoding_analysis_single_electrode(
        epochs, stride, window_size):
    # global_tmin = 0. # min([epoch.times.min() for epoch in epochs.values()])
    # global_tmax = max([epoch.times.max() for epoch in epochs.values()])
    # windows_left = np.arange(global_tmin, global_tmax, stride)
    # windows_right = np.minimum(global_tmax, windows_left + window_size)
    # windows = list(zip(windows_left, windows_right))

    # `outcomes` stores prediction outcomes for each epoch under the optimal model
    outcomes = {}
    # `test_scores` stores cross-validated estimates of held-out generalization
    train_scores, test_scores = {}, {}
    # `models` stores the fitted models
    models = {}

    phoneme_pairs = next(iter(epochs.values())).metadata.phoneme_pair.unique()

    electrodes = electrode_df.query("speech_responsive").reset_index()

    for _, row in tqdm(electrodes.iterrows(), total=len(electrodes)):
        for phoneme_pair in phoneme_pairs:
            epochs_ij = epochs[row.subject]
            # manual filtering for performance
            selection = epochs_ij.metadata.phoneme_pair == phoneme_pair

            if selection.sum() == 0:
                continue

            # num_trials * num_times
            X = epochs_ij.get_data(picks=[row.electrode_idx])[selection].squeeze(1)
            y = (epochs_ij.metadata.word_end.str[0] == phoneme_pair[0])[selection].values
            # stratify_class = epochs_ij.metadata.stratify_class[selection].values

            ####

            cv_inner = StratifiedKFold(3, shuffle=True, random_state=42)
            cv_outer = StratifiedKFold(3, shuffle=True, random_state=42)

            Cs = np.logspace(-6, 6, 6)

            pipeline = [StandardScaler()]
            pipeline.append(LogisticRegressionCV(
                Cs=Cs, cv=cv_inner, max_iter=10000, n_jobs=1,
                solver="liblinear"))
            model = make_pipeline(*pipeline)
            fitted = cross_validate(model, X, y, cv=cv_outer, scoring="roc_auc",
                                    return_estimator=True,
                                    return_train_score=True,
                                    n_jobs=2)

            result_key = (row.subject, row.electrode_idx, phoneme_pair)

            train_scores[result_key] = fitted["train_score"]
            test_scores[result_key] = fitted["test_score"]

            # onl store outcomes on test folds
            outcomes[result_key] = pd.concat([
                pd.DataFrame({"decoder_target": y[test_idxs],
                              "decoder_prediction": estimator.predict(X[test_idxs]),
                              "decoder_proba": estimator.predict_proba(X[test_idxs])[:, 1],
                              "fold": fold},
                             index=test_idxs)
                for fold, ((_, test_idxs), estimator) in enumerate(zip(cv_outer.split(X, y), fitted["estimator"]))
            ])

            models[result_key] = fitted["estimator"]

    return train_scores, test_scores, outcomes, models

In [198]:
from threadpoolctl import threadpool_limits

with threadpool_limits(limits=1):
    all_train_scores, all_scores, all_outcomes, all_models = \
        run_decoding_analysis_single_electrode(epochs, stride=1.0, window_size=1.0)

  0%|          | 0/1074 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
train_scores_df = pd.concat(
        {key: pd.Series(scores_i).rename("roc_auc") for key, scores_i in all_train_scores.items()},
        names=["subject", "electrode_idx", "phoneme_pair", "fold"])
train_scores_df

In [ ]:
scores_df = pd.concat(
        {key: pd.Series(scores_i).rename("roc_auc") for key, scores_i in all_scores.items()},
        names=["subject", "electrode_idx", "phoneme_pair", "fold"])
scores_df

In [169]:
hparam_df = pd.DataFrame(
    [{"subject": subject, "electrode_idx": electrode_idx,
      "phoneme_pair": phoneme_pair,
      "fold": j,
      "C": fold_fit.steps[-1][1].C_[0]}
     for (subject, electrode_idx, phoneme_pair), fitted_i in all_models.items()
     for j, fold_fit in enumerate(fitted_i)])
hparam_df = hparam_df.set_index(["subject", "electrode_idx", "phoneme_pair", "fold"])

In [170]:
merged_df = pd.merge(scores_df, hparam_df, left_index=True, right_index=True, how="left", validate="1:1")
merged_df["log_C"] = np.log10(merged_df["C"])

In [ ]:
merged_df

In [ ]:
sns.histplot(data=merged_df, x="log_C", multiple="dodge")

In [186]:
train_scores_df.to_csv(Path(outdir) / "train_scores.csv")
scores_df.to_csv(Path(outdir) / "scores.csv")

In [174]:
torch.save({"models": all_models, "outcomes": all_outcomes}, Path(outdir) / "outcomes.pt")

In [ ]:
sns.catplot(data=scores_df.reset_index(), x="subject", y="roc_auc", kind="box", aspect=2)